# LLM-Enhanced Momentum Strategy

This notebook implements the LLM-enhanced momentum strategy described in the paper "ChatGPT in Systematic Investing -- Enhancing Risk-Adjusted Returns with LLMs" by Nikolas Anic, Andrea Barbon, Ralf Seiz, and Carlo Zarattini. The strategy combines cross-sectional momentum with predictive signals extracted from firm-specific news using a large language model.

**Paper Citation:**
Anic, N., Barbon, A., Seiz, R., & Zarattini, C. (2025). *ChatGPT in Systematic Investing -- Enhancing Risk-Adjusted Returns with LLMs*. arXiv preprint arXiv:2510.26228.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
MOMENTUM_PERIOD = 126
HOLDING_PERIOD = 21
CAPITAL = 1000000
TRANSACTION_COST = 0.001

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download price data
prices = yf.download(UNIVERSE, start='2010-01-01', end='2023-10-30', group_by='ticker')['Adj Close']

# Compute momentum factor
momentum = prices.pct_change(MOMENTUM_PERIOD).shift()

# Cross-sectional normalization
momentum_zscore = (momentum - momentum.mean(axis=1, skipna=True).values.reshape(-1, 1)) / momentum.std(axis=1, skipna=True).values.reshape(-1, 1)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Generate signals
signals = momentum_zscore.rank(axis=1, pct=True, ascending=False)

# Position sizing
weights = signals / signals.sum(axis=1, skipna=True).values.reshape(-1, 1)

# Portfolio construction
positions = weights.shift() * CAPITAL

## Phase 4 — Vectorized Backtest

In [ ]:
# Compute daily returns
daily_returns = prices.pct_change().shift(-1)

# Compute portfolio returns
portfolio_returns = (daily_returns * positions).sum(axis=1)

# Apply transaction costs
portfolio_returns -= TRANSACTION_COST * np.abs(positions.diff()).sum(axis=1)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Compute performance metrics
cumulative_returns = (1 + portfolio_returns).cumprod()
sharpe = np.sqrt(252) * portfolio_returns.mean() / portfolio_returns.std()
sortino = np.sqrt(252) * portfolio_returns.mean() / portfolio_returns[portfolio_returns < 0].std()
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar = sharpe / abs(max_drawdown)

# Print metrics
print(f'Sharpe: {sharpe:.2f}')
print(f'Sortino: {sortino:.2f}')
print(f'Calmar: {calmar:.2f}')
print(f'Max Drawdown: {max_drawdown*100:.2f}%%')

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor(prices, positions):
    daily_pnl = (prices.pct_change() * positions).sum(axis=1)
    print(f'Daily P&L: {daily_pnl[-1]:.2f}')
    print('Current Positions:')
    print(positions.iloc[-1].dropna())

# Example usage
monitor(prices, positions)